In [44]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

## 샤프 계산

In [49]:
import pandas as pd
import numpy as np
from pprint import pprint

df_test = pd.read_csv('./data/test_future.csv') ## 2023-05-31 ~ 2023-06-21
cc_close = df_test.sort_values(by='일자').groupby( by=['종목코드'])['종가'].apply(list).to_dict()

df_submission = pd.read_csv('./sub/앙상블_최종.csv')
index_to_drop = df_submission[df_submission['종목코드'].isin(['A038530', 'A079810'])].index
df_submission = df_submission.drop(index_to_drop, axis=0)

cc_rank = df_submission.sort_values(by='순위')['종목코드'].to_list()

In [50]:
np_cc_buy = np.array([ cc_close[cc] for cc in cc_rank[:200] ])
np_cc_sell = np.array([ cc_close[cc] for cc in cc_rank[-200:] ])

assert np_cc_buy.shape[1] == 15

value = {}
value['매매일수'] = 15
value['무위험수익률'] = 0.035
value['총 매수 수익률']  = + np.sum( np_cc_buy [:,-1]/np_cc_buy [:,0]-1.0 )
value['총 공매도 수익률'] = - np.sum( np_cc_sell[:,-1]/np_cc_sell[:,0]-1.0 )
value['총 자산 최종 수익률'] = ( value['총 매수 수익률'] + value['총 공매도 수익률'] ) / 400
value['연율화된 총자산 최종 수익률'] = value['총 자산 최종 수익률'] * 250 / value['매매일수']

profit_day_buy = + ( np_cc_buy [:,1:]/np_cc_buy [:,:-1]-1.0 )
profit_day_sell = - ( np_cc_sell[:,1:]/np_cc_sell[:,:-1]-1.0 )

value['연율화된 일간수익률의 일별 평균'] = ( profit_day_buy + profit_day_sell ).sum(0) * 250 / 400
value['연율화된 일간수익률의 평균'] = np.mean(value['연율화된 일간수익률의 일별 평균'])
value['총자산 일간 수익률 변동성'] = ( np.sum( ( value['연율화된 일간수익률의 일별 평균'] - value['연율화된 일간수익률의 평균'] ) ** 2.0 ) / (15-2) ) ** 0.5
value['샤프지수'] = ( value['연율화된 총자산 최종 수익률'] - value['무위험수익률'] ) / value['총자산 일간 수익률 변동성']

pprint(value)

print('총자산 일간 수익률 변동성(다른계산법들)', 
   value['연율화된 일간수익률의 일별 평균'].std() * np.sqrt(14/13),
   np.concatenate( ( profit_day_buy, profit_day_sell ) ).mean(0).std() * np.sqrt(14/13) * 250
   )

{'매매일수': 15,
 '무위험수익률': 0.035,
 '샤프지수': -0.1761622050240123,
 '연율화된 일간수익률의 일별 평균': array([ 0.1719661 , -0.33788374, -0.85963809, -0.24943032,  0.48784975,
       -0.16895455, -0.12722673,  1.07884683, -0.24316125,  1.33604108,
       -0.42060572, -1.27581296,  0.82649915, -1.63933806]),
 '연율화된 일간수익률의 평균': -0.10148918076283704,
 '연율화된 총자산 최종 수익률': -0.1137312544400136,
 '총 공매도 수익률': -12.37191730336273,
 '총 매수 수익률': 9.642367196802404,
 '총 자산 최종 수익률': -0.006823875266400816,
 '총자산 일간 수익률 변동성': 0.8442858354307065}
총자산 일간 수익률 변동성(다른계산법들) 0.8442858354307065 0.8442858354307068


---